In [ ]:
import os
import polars as pl
import numpy as np
import random

from collections import defaultdict
from typing import Literal, List, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [ ]:
class AveragePool(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, masks=None):
        B, N, D = x.shape

        if masks is not None:
            return x.sum(dim=1) / masks.float().sum(dim=1).unsqueeze(1)
        return x.mean(dim=1)
    
class GatedPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

    def forward(self, x, mask=None):
        B, N, D = x.shape
        gate_weights = self.gate_mlp(x)
        
        gated_features = x * gate_weights

        if mask is not None:
            gated_features = gated_features * mask.unsqueeze(-1)

        return torch.sum(gated_features, dim=1)
    
class SoftAttentionPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.attention_weights_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1)
        )

    def forward(self, x, mask=None):
        B, N, D = x.shape
        raw_scores = self.attention_weights_mlp(x)

        if mask is not None:
            raw_scores = raw_scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        
        attention_weights = torch.softmax(raw_scores, dim=1)

        weighted_embeddings = x * attention_weights

        return torch.sum(weighted_embeddings, dim=1)

class AttentionPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.query = nn.Parameter(torch.zeros(1, embed_dim))
        
        self.attention_net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Tanh(),
            nn.Linear(embed_dim, 1)
        )
        
        nn.init.xavier_uniform_(self.query)

    def forward(self, x, mask=None):
        B, N, D = x.shape
        
        query_expanded = self.query.expand(B, N, -1)
        
        combined = x + query_expanded

        raw_scores = self.attention_net(combined)

        if mask is not None:
            raw_scores = raw_scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        
        attention_weights = torch.softmax(raw_scores, dim=1)
        weighted_sum = torch.sum(x * attention_weights, dim=1)

        return weighted_sum

In [ ]:
class Classifier(nn.Module):
    def __init__(self, pooled_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(pooled_dim, num_classes)

    def forward(self, pooled_embedding):
        logits = self.fc(pooled_embedding)
        return logits
    
class Regressor(nn.Module):
    def __init__(self, pooled_dim: int):
        super().__init__()
        self.fc = nn.Linear(pooled_dim, 1)

    def forward(self, pooled_embedding):
        prediction = self.fc(pooled_embedding)
        return prediction

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, patient_ids, id_to_path, labels):
        self.patient_ids = patient_ids
        self.id_to_path = id_to_path
        self.labels = labels

    def __len__(self):
        return len(self.patient_ids)
    
    def get_labels(self):
        return [self.labels[x] for x in self.patient_ids]

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        embedding_path = self.id_to_path[patient_id]
        
        embedding_data = torch.load(embedding_path)
        embeddings = embedding_data["cls"]
        label = float(self.labels[patient_id])

        return embeddings, label

In [ ]:
class StratifiedSampler(Sampler):
    def __init__(self, labels, batch_size):
        self.labels = labels
        self.batch_size = batch_size
        
        self.label_to_indices = defaultdict(list)
        for idx, label in enumerate(self.labels):
            self.label_to_indices[label].append(idx)
        
        self.unique_labels = sorted(self.label_to_indices.keys())
        self.n_classes = len(self.unique_labels)
        
        self.n_samples_per_class = self.batch_size // self.n_classes
        self.remainder_samples = self.batch_size % self.n_classes
        
        self.min_samples_per_class = min(len(indices) for indices in self.label_to_indices.values())
        self.n_batches = int(self.min_samples_per_class / (self.n_samples_per_class + (1 if self.remainder_samples > 0 else 0)))

        if self.n_batches == 0 and self.min_samples_per_class > 0:
            self.n_batches = 1

    def __iter__(self):
        shuffled_indices = {label: random.sample(indices, len(indices))
                            for label, indices in self.label_to_indices.items()}
        
        for i in range(self.n_batches):
            batch = []
            
            for label in self.unique_labels:
                start_idx = i * self.n_samples_per_class
                end_idx = start_idx + self.n_samples_per_class
                class_indices = shuffled_indices[label]
                for j in range(start_idx, end_idx):
                    batch.append(class_indices[j % len(class_indices)])

            if self.remainder_samples > 0:
                remainder_indices = []
                for k in range(self.remainder_samples):
                    label_to_add = self.unique_labels[k]
                    start_idx = i * self.n_samples_per_class
                    class_indices = shuffled_indices[label_to_add]
                    
                    rem_idx = (start_idx + self.n_samples_per_class) % len(class_indices)
                    remainder_indices.append(class_indices[rem_idx])
                batch.extend(remainder_indices)
            
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.n_batches

In [ ]:
def pad_collate_fn(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    
    embeddings_list, labels_list = zip(*batch)
    
    max_len = embeddings_list[0].shape[0]
    padded_embeddings = torch.zeros(len(embeddings_list), max_len, embeddings_list[0].shape[1])
    masks = torch.zeros(len(embeddings_list), max_len, dtype=torch.bool)
    
    for i, embedding in enumerate(embeddings_list):
        seq_len = embedding.shape[0]
        padded_embeddings[i, :seq_len, :] = embedding
        masks[i, :seq_len] = True
        
    labels = torch.tensor(labels_list, dtype=torch.float32).unsqueeze(1)
    
    return padded_embeddings, labels, masks

In [ ]:
embeddings_path = "/scratch/VM/radio-foundation/cache/embeddings/NSCLC_Radiomics"

files = [x for x in os.listdir(embeddings_path) if x.endswith(".pth")]
id_to_path = {
    f.replace(".pth", "") : os.path.join(embeddings_path, f) for f in files
}

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets"

clinical_raw = pl.read_csv(os.path.join(data_path, "NSCLC-Radiomics/clinical.csv"))
clinical_raw.head()

In [ ]:
label_name = "Clinical.N.Stage"
clinical_raw[label_name].value_counts()

In [ ]:
df = clinical_raw#.filter(pl.col(label_name) != 'NA')
labels = {key: float(value) for key, value in zip(df['PatientID'], df[label_name]) if value in [0, 1, 2, 3]}

fmean = np.mean(list(labels.values()))
fstd = np.std(list(labels.values()))

labels_norm = {k: (v - fmean) / fstd for k, v in labels.items()}
print(fmean, fstd)

In [ ]:
patient_ids = list(labels_norm.keys())
exist_patient_ids = [x for x in patient_ids if x in id_to_path.keys()]
train_ids, val_ids = train_test_split(exist_patient_ids, test_size=0.2, random_state=4)

train_dataset = EmbeddingDataset(train_ids, id_to_path, labels_norm)
val_dataset = EmbeddingDataset(val_ids, id_to_path, labels_norm)

In [ ]:
class PoolingRegressor(nn.Module):
    def __init__(self, pool_method: Literal["average", "gated", "soft_attention", "attention"], embed_dim: int):
        super().__init__()
        
        if pool_method == "average":
            self.pooler = AveragePool()
        elif pool_method == "gated":
            self.pooler = GatedPool(embed_dim)
        elif pool_method == "soft_attention":
            self.pooler = SoftAttentionPool(embed_dim)
        elif pool_method == "attention":
            self.pooler = AttentionPool(embed_dim)
        else:
            raise ValueError(f"Unknown pooling method: {pool_method}")

        self.regressor = Regressor(pooled_dim=embed_dim)

    def forward(self, embeddings, mask=None):
        pooled_embedding = self.pooler(embeddings, mask)
        prediction = self.regressor(pooled_embedding)
        return prediction

In [ ]:
def do_train(
        model,
        optimizer,
        loss_fn,
        train_dataloader,
        val_dataloader,
        num_epochs,
        device,
        verbose=False
    ):
    train_loss_list = []
    val_loss_list = []

    best_val_loss = float("inf")
    best_model_state = model.state_dict()
    
    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        for embeddings, labels, masks in train_dataloader:
            embeddings, labels, masks = embeddings.to(device), labels.to(device), masks.to(device)
            
            predictions = model(embeddings, masks)
            loss = loss_fn(predictions, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * embeddings.size(0)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for embeddings, labels, masks in val_dataloader:
                embeddings, labels, masks = embeddings.to(device), labels.to(device), masks.to(device)
                
                predictions = model(embeddings, masks)
                loss = loss_fn(predictions, labels)
                
                val_loss += loss.item() * embeddings.size(0)

        avg_train_loss = (train_loss / len(train_dataset)) * fstd
        avg_val_loss = (val_loss / len(val_dataset)) * fstd

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()

        train_loss_list.append(avg_train_loss)
        val_loss_list.append(avg_val_loss)
        
        if epoch % 10 == 0 and verbose:
            print(f"Epoch [{epoch+1}/{num_epochs}], "
                    f"Training MSE: {avg_train_loss:.4f}, "
                    f"Validation MSE: {avg_val_loss:.4f}")
            
    return train_loss_list, val_loss_list, best_model_state

In [ ]:
def get_predictions(model, dataloader, device, fmean, fstd):
    all_labels = []
    all_predictions = []
    model.eval()
    with torch.no_grad():
        for embeddings, labels, masks in dataloader:
            embeddings, labels, masks = embeddings.to(device), labels.to(device), masks.to(device)
            
            predictions = model(embeddings, masks)
            all_labels.append(labels.cpu() * fstd + fmean)
            all_predictions.append(predictions.cpu() * fstd + fmean)

    all_predictions = torch.cat(all_predictions, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_labels, all_predictions

In [ ]:
EMBED_DIM = 768
num_epochs = 100
batch_size = 32
learning_rate = 0.0001
pooling_method = "average"

#train_sampler = StratifiedSampler(labels=train_dataset.get_labels(), batch_size=batch_size)

train_dataloader = DataLoader(
    train_dataset,
    shuffle=False,
    collate_fn=pad_collate_fn,
    batch_size=batch_size,
    #batch_sampler=train_sampler
)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=pad_collate_fn)

device = torch.device("cuda")
model = PoolingRegressor(pool_method=pooling_method, embed_dim=EMBED_DIM).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

train_loss_list, val_loss_list, best_model_state = do_train(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device
)
model.load_state_dict(best_model_state)


In [ ]:
val_vmin = min(val_loss_list)

plt.axhline(y=val_vmin, color='k', linestyle=':', alpha=0.5)
plt.plot(train_loss_list, label="Train")
plt.plot(val_loss_list, label="Validation")
plt.grid(True)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, device, fmean, fstd)

plt.scatter(all_labels, all_predictions)
plt.xlabel("True Labels")
plt.ylabel("Predicted Labels")
plt.xticks([1, 2, 3, 4])
plt.yticks([1, 2, 3, 4])
plt.grid(True)
plt.show()